# Week 5 – LSTM Model Training

In this notebook, an LSTM model is implemented
to forecast energy consumption using time-series data.


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


In [2]:
# Load dataset
df = pd.read_csv("smart_home_energy_consumption_large.csv")

# Combine Date and Time
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df = df.set_index('Datetime')

# Drop unused columns
df = df.drop(columns=['Date', 'Time'])

energy_col = "Energy Consumption (kWh)"

# Sort by time
df = df.sort_index()

# Keep only energy column for LSTM (univariate)
energy_data = df[[energy_col]]


In [3]:
scaler = MinMaxScaler(feature_range=(0, 1))
energy_scaled = scaler.fit_transform(energy_data.values)


In [4]:
TIME_STEPS = 24

def create_sequences(data, time_steps):
    X, y = [], []
    for i in range(time_steps, len(data)):
        X.append(data[i-time_steps:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X, y = create_sequences(energy_scaled, TIME_STEPS)

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (99976, 24)
y shape: (99976,)


In [5]:
split_index = int(len(X) * 0.8)

X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

# Reshape for LSTM (3D input)
X_train = X_train.reshape((X_train.shape[0], TIME_STEPS, 1))
X_test = X_test.reshape((X_test.shape[0], TIME_STEPS, 1))

print("X_train shape:", X_train.shape)


X_train shape: (79980, 24, 1)


In [6]:
model = Sequential()

model.add(LSTM(64, return_sequences=True, input_shape=(TIME_STEPS, 1)))
model.add(Dropout(0.2))

model.add(LSTM(32))
model.add(Dropout(0.2))

model.add(Dense(1))

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 24, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,345 (114.63 KB)

 Trainable params: 29,345 (114.63 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mean_squared_error'
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop]
)


Epoch 1/30
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 55s 23ms/step - loss: 0.0581 - val_loss: 0.0580
Epoch 2/30
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 52s 23ms/step - loss: 0.0587 - val_loss: 0.0580
Epoch 3/30
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 82s 23ms/step - loss: 0.0582 - val_loss: 0.0581
Epoch 4/30
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 50s 22ms/step - loss: 0.0590 - val_loss: 0.0580
Epoch 5/30
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 51s 23ms/step - loss: 0.0575 - val_loss: 0.0580
Epoch 6/30
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 81s 22ms/step - loss: 0.0587 - val_loss: 0.0580
Epoch 7/30
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 49s 22ms/step - loss: 0.0583 - val_loss: 0.0580
Epoch 8/30
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 49s 22ms/step - loss: 0.0582 - val_loss: 0.0580
Epoch 9/30
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 82s 22ms/step - loss: 0.0586 - val_loss: 0.0580
Epoch 10/30
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 83s 22ms/step - loss: 0.0578 - val_loss: 0.0581
Epoch 11/30
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 80s 22ms/step - loss: 0.0594 - val_loss: 0.0580
Epoch 12